# Antibacterial molecules — EDA

~40,000 small molecules screened for antibacterial activity against gonorrhea,
~97% inactive (`docs/plan.md` 3.1) — the core case study for the negative-data
question. This notebook covers class balance, which RDKit descriptors best
separate active from inactive molecules (a leakage smell-test), how those
descriptors correlate with each other, and how the hit rate varies by scaffold
family (the grouping a harder scaffold split holds out).

The plotting helpers live in `negative_ml.viz.diagnostics`, so this notebook and
`scripts/molecules_eda.py` draw the same figures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from negative_ml.molecules import io as molecules
from negative_ml.molecules.config import (
    MOLECULE_TRAIN_STEM,
    MOLECULES_EDA_TOP_FEATURES,
    MOLECULES_EDA_TOP_SCAFFOLDS,
)
from negative_ml.viz import diagnostics as diag
from negative_ml.viz.style import INK, LIGHT_GRAY, PALETTE, style_axes

# rdkit descriptors are used (rather than morgan fingerprints or both) so the
# EDA figures below are labeled with named, interpretable features.
ds = molecules.load_screen(MOLECULE_TRAIN_STEM, "rdkit")
n_pos = int(ds.y.sum())
print(
    f"{ds.n_samples:,} molecules; {n_pos:,} active ({ds.y.mean():.2%} minority); "
    f"{ds.X.shape[1]} features"
)

## 1. Class balance

Only ~3% of the screen is active; this stark imbalance is the reason the study
exists.

In [ ]:
n_neg = ds.n_samples - n_pos
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["inactive (majority)", "active (minority)"],
    [n_neg, n_pos],
    color=[LIGHT_GRAY, PALETTE["magenta"]],
)
for bar, count in zip(bars, [n_neg, n_pos]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{count:,}\n({count / ds.n_samples:.1%})",
        ha="center",
        va="bottom",
        fontsize=9,
        color=INK,
    )
ax.set_ylabel("molecules")
ax.set_title("Class balance (antibacterial screen)")
style_axes(ax)
plt.show()

## 2. Which descriptors best separate active from inactive? (leakage
smell-test)

Pearson and Spearman correlation of each RDKit descriptor with the hit label. A
near-perfect correlation would flag a leaked feature; the top descriptors here
sit in a healthy, moderate range.

In [ ]:
label_corr = diag.label_correlation_table(ds.X, ds.y, ds.feature_names)
top_corr = label_corr.head(MOLECULES_EDA_TOP_FEATURES)

fig, ax = plt.subplots(figsize=(10, 4.5))
diag.plot_label_correlations(top_corr, ax)
ax.set_title(
    f"Descriptor-activity correlation (top {len(top_corr)} by |Pearson|, "
    "leakage smell-test)"
)
plt.tight_layout()
plt.show()

## 3. Feature distributions by class

The same top descriptors, now split by class. Panels where the active
(magenta) and inactive (gray) outlines diverge are the features actually
driving separation, complementing the linear correlation view above.

In [ ]:
top_features = top_corr["feature"].tolist()
feature_idx = [ds.feature_names.index(f) for f in top_features]
X_top = ds.X[:, feature_idx]

fig = diag.plot_class_stratified_distributions(
    X_top, ds.y, top_features, minority_label="active", majority_label="inactive"
)
fig.suptitle(
    f"Top {len(top_features)} features by label correlation — active vs. inactive",
    fontsize=13,
)
plt.show()

## 4. Descriptor correlation structure

Pairwise Pearson correlation among the highest-variance descriptors. Chemistry
descriptors are often redundant (e.g. molecular weight correlates with several
size-related counts), which matters for methods like the per-feature negative
control that treat features independently.

In [ ]:
variances = np.var(ds.X, axis=0)
top_var_idx = np.argsort(variances)[::-1][:MOLECULES_EDA_TOP_FEATURES]
top_var_names = [ds.feature_names[i] for i in top_var_idx]
feature_corr = diag.feature_correlation_matrix(ds.X[:, top_var_idx], top_var_names)

fig, ax = plt.subplots(figsize=(10, 9))
diag.plot_correlation_heatmap(
    feature_corr,
    ax,
    title=f"Descriptor correlation (top {MOLECULES_EDA_TOP_FEATURES} by variance)",
)
plt.tight_layout()
plt.show()

## 5. Activity rate by scaffold

Molecules sharing a Bemis-Murcko scaffold are grouped, since a scaffold split
(whole structural families held out together) is the harder, more realistic
chemistry evaluation. The largest scaffold families range from all-inactive to
well above the ~3% baseline rate — exactly the distribution shift a scaffold
split is meant to test.

In [ ]:
per_scaffold = diag.per_group_positive_rate(ds.groups, ds.y)
named = per_scaffold["group"].notna() & (per_scaffold["group"] != "")
large_scaffolds = per_scaffold[named & (per_scaffold["n"] >= 2)].copy()
if len(large_scaffolds) > MOLECULES_EDA_TOP_SCAFFOLDS:
    large_scaffolds = large_scaffolds.nlargest(MOLECULES_EDA_TOP_SCAFFOLDS, "n")

fig, ax = plt.subplots(figsize=(max(10, 0.4 * len(large_scaffolds)), 4.5))
diag.plot_per_group_rate(large_scaffolds, ax, label="active")
ax.set_title(
    f"Activity rate by scaffold (top {len(large_scaffolds)} by molecule count)"
)
plt.tight_layout()
plt.show()